In [1]:
import polars as pl
import json
import pandas as pd

# Load CSV
df = pl.read_csv("/home/mahdi/Loan_Reommender_System/wizard_solution/MEC-LoanRecomn_Scenarios-V0.7.csv", null_values=["nan"])

df_1 = pd.read_csv("/home/mahdi/Loan_Reommender_System/wizard_solution/MEC-LoanRecomn_Scenarios-V0.7.csv")
# Convert to list of dictionaries (row-oriented)
loans = df.to_dicts()
print(len(loans))

# Dump to JSON string
json_str = json.dumps(loans, indent=4)

# Save to file
with open("output.json", "w") as list_loans:
    list_loans.write(json_str)



211


In [2]:
df_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   alias                   211 non-null    object 
 1   package_name            211 non-null    object 
 2   contract_type           211 non-null    object 
 3   granted_method          211 non-null    object 
 4   loan_amount_limit       211 non-null    int64  
 5   deposit_duration        208 non-null    float64
 6   interest_rate           211 non-null    int64  
 7   repayment_duration      211 non-null    int64  
 8   loan_coefficient        208 non-null    float64
 9   credit_score            211 non-null    object 
 10  minimum_deposit_amount  3 non-null      float64
 11  maximum_deposit_amount  143 non-null    float64
 12  minimum_loan_amount     211 non-null    int64  
 13  guarantee               211 non-null    int64  
 14  receiving_channel       211 non-null    ob

In [26]:
def calculate_monthly_repayment(loan_amount, interest_rate, repayment_duration):

    # Convert annual interest rate to monthly interest rate (decimal)
    # monthly_repayment = (4 / (12*100))
    # interest_rate =23
    monthly_interest_rate = (interest_rate / (12*100))
    

    # Apply the formula from the Excel file
    numerator = loan_amount * monthly_interest_rate * ((1 + monthly_interest_rate) ** repayment_duration)
    denominator = ((1 + monthly_interest_rate) ** repayment_duration) - 1
    monthly_repayment = numerator / denominator
    # monthly_repayment = monthly_interest_rate 

    return round(monthly_repayment)

# Example usage
loan_amount = 50000  # 1 billion
interest_rate = 23  # 23%
repayment_duration = 12  # 36 months

monthly_repayment = calculate_monthly_repayment(loan_amount, interest_rate, repayment_duration)


print(f"Monthly repayment amount: {monthly_repayment}")
# print(f"Total repayment amount: {total_repayment:,.2f}")

Monthly repayment amount: 4704


In [4]:
(23 / (12*100))

0.019166666666666665

In [5]:
def filter_loans(loans, deposit_amount=None, deposit_duration=None, 
                 loan_amount=None, credit_score=None, interest_rate=None, 
                 repayment_duration=None):
    
    params = {
        "deposit_amount": deposit_amount,
        "deposit_duration": deposit_duration,
        "loan_amount": loan_amount,  
        "credit_score": credit_score,
        "interest_rate": interest_rate,
        "repayment_duration": repayment_duration,
    }

    filtered_loans = []

    for loan in loans:
        loan_copy = loan.copy()  # Avoid mutating the original loan data
        
        # Step 1: Apply the loan_amount logic
        if loan_amount is not None:
            loan_copy["loan_amount"] = loan_amount
            
            # Check that loan_coefficient exists and is not None or zero
            coefficient = loan_copy.get('loan_coefficient')
            rd = loan_copy.get('repayment_duration')
            ir = loan_copy.get('interest_rate')
            la = loan_copy.get('loan_amount')


            if coefficient:
                loan_copy["deposit_amount"] = loan_amount / coefficient
            else:
                loan_copy["deposit_amount"] = None  # Or set to 0 or skip, depending on your logic

            loan_copy["repayment_amount"] = calculate_monthly_repayment(la, ir, rd)

        # Step 2: Check conditions
        conditions = []  
        for key, value in params.items():
            if value is not None:
                if key == "loan_amount":
                    # Check if loan's limit is enough
                    conditions.append(loan.get("loan_amount_limit", 0) >= value)
                else:
                    conditions.append(loan_copy.get(key) == value)

        if conditions and all(conditions):
            filtered_loans.append(loan_copy)
    
    return filtered_loans


In [6]:
filtered = filter_loans(loans, loan_amount= 1000000000)
print(len(filtered))
print("Filtered loans:")
print(filtered[1])
# for loan in filtered:
#     print(loan)

211
Filtered loans:
{'alias': 'شایان', 'package_name': 'شایان یک', 'contract_type': 'جعاله', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 3000000000, 'deposit_duration': 1, 'interest_rate': 23, 'repayment_duration': 12, 'loan_coefficient': 50, 'credit_score': 'A', 'minimum_deposit_amount': None, 'maximum_deposit_amount': 15000000000, 'minimum_loan_amount': 100000000, 'guarantee': 1, 'receiving_channel': 'غیر حضوری/ حضوری', 'account_type': None, 'loan_amount': 1000000000, 'deposit_amount': 20000000.0, 'repayment_amount': 94076321}


## Fill features first 

In [23]:
def fill_calculated_param (loans, loan_amount = None, deposit_amount= None):
    
    updated_loans = []
    
    for loan in loans:

        coefficient = loan.get('loan_coefficient')

        if (loan_amount is not None and deposit_amount is not None) or (loan_amount is not None and deposit_amount is None):
            loan["loan_amount"] = loan_amount
            if coefficient:
                loan["deposit_amount"] = round(loan_amount / coefficient)
            else:
                loan["deposit_amount"] = None

        elif deposit_amount is not None and loan_amount is None:
            loan["deposit_amount"] = deposit_amount
            if coefficient:
                loan["loan_amount"] = round(deposit_amount * coefficient)
            else:
                loan["loan_amount"] = None
            
        else:
            loan["loan_amount"] = None
            loan["deposit_amount"] = None
        
        rd = loan.get('repayment_duration')
        ir = loan.get('interest_rate')
        la = loan.get('loan_amount')

        if la is not None and ir is not None and rd is not None:
            loan["repayment_amount"] = calculate_monthly_repayment(la, ir, rd)
        else:
            loan["repayment_amount"] = None

        

        updated_loans.append(loan)

    return updated_loans

In [24]:
filtered = fill_calculated_param(loans, loan_amount= 1000000000, deposit_amount= 12000000)
print(len(filtered))
print("Filtered loans:")
print(filtered[1])
for loan in filtered:
    print(loan)

211
Filtered loans:
{'alias': 'شایان', 'package_name': 'شایان یک', 'contract_type': 'جعاله', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 3000000000, 'deposit_duration': 1, 'interest_rate': 23, 'repayment_duration': 12, 'loan_coefficient': 50, 'credit_score': 'A', 'minimum_deposit_amount': None, 'maximum_deposit_amount': 15000000000, 'minimum_loan_amount': 100000000, 'guarantee': 1, 'receiving_channel': 'غیر حضوری/ حضوری', 'account_type': None, 'loan_amount': 1000000000, 'deposit_amount': 20000000, 'repayment_amount': 94076321}
{'alias': 'شایان', 'package_name': 'شایان یک', 'contract_type': 'مرابحه', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 3000000000, 'deposit_duration': 1, 'interest_rate': 23, 'repayment_duration': 12, 'loan_coefficient': 50, 'credit_score': 'A', 'minimum_deposit_amount': None, 'maximum_deposit_amount': 15000000000, 'minimum_loan_amount': 100000000, 'guarantee': 1, 'receiving_channel': 'غیر حضوری/ حضوری', 'account_type': None, 'loan_amount':

In [9]:


def filter_loans(loans, deposit_amount=None, deposit_duration=None, 
                loan_amount=None, credit_score=None, interest_rate=None, 
                repayment_duration=None):

    params = {
        "deposit_amount": deposit_amount,
        "deposit_duration": deposit_duration,
        "loan_amount": loan_amount,  
        "credit_score": credit_score,
        "interest_rate": interest_rate,
        "repayment_duration": repayment_duration,
    }


    # Step 1: Update all loans with loan_amount logic
    updated_loans = fill_calculated_param (loans, loan_amount, deposit_amount)

    # Step 2: Filter based on updated data
    filtered_loans = []
    for loan in updated_loans:
        conditions = []
        for key, value in params.items():
            if value is not None:
                if key == "loan_amount":
                    # Check against the limit only, not the updated value
                    conditions.append(loan.get("loan_amount_limit", 0) >= value)
                else:
                    conditions.append(loan.get(key) == value)

        if conditions and all(conditions):
            filtered_loans.append(loan)

    return filtered_loans


In [19]:
# filtered = filter_loans(loans, loan_amount= 1000000000, deposit_amount= 1200000000)
filtered = filter_loans(loans, deposit_amount= 1200000000)
# filtered = filter_loans(loans, loan_amount= 1000000000)
print(len(filtered))
print("Filtered loans:")
print(filtered[1])
for loan in filtered:
    print(loan)

211
Filtered loans:
{'alias': 'شایان', 'package_name': 'شایان یک', 'contract_type': 'جعاله', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 3000000000, 'deposit_duration': 1, 'interest_rate': 23, 'repayment_duration': 12, 'loan_coefficient': 50, 'credit_score': 'A', 'minimum_deposit_amount': None, 'maximum_deposit_amount': 15000000000, 'minimum_loan_amount': 100000000, 'guarantee': 1, 'receiving_channel': 'غیر حضوری/ حضوری', 'account_type': None, 'loan_amount': 60000000000, 'deposit_amount': 1200000000, 'repayment_amount': 5644579280}
{'alias': 'شایان', 'package_name': 'شایان یک', 'contract_type': 'مرابحه', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 3000000000, 'deposit_duration': 1, 'interest_rate': 23, 'repayment_duration': 12, 'loan_coefficient': 50, 'credit_score': 'A', 'minimum_deposit_amount': None, 'maximum_deposit_amount': 15000000000, 'minimum_loan_amount': 100000000, 'guarantee': 1, 'receiving_channel': 'غیر حضوری/ حضوری', 'account_type': None, 'loan_amo

In [20]:
filtered_df= pd.DataFrame(filtered)

In [21]:
filtered_df.head()

,alias,package_name,contract_type,granted_method,loan_amount_limit,deposit_duration,interest_rate,repayment_duration,loan_coefficient,credit_score,minimum_deposit_amount,maximum_deposit_amount,minimum_loan_amount,guarantee,receiving_channel,account_type,loan_amount,deposit_amount,repayment_amount
0,شایان,شایان یک,مرابحه,واریز به حساب,3000000000,1.0,23,12,50.0,A,None,1.500000e+10,100000000,1,غیر حضوری/ حضوری,None,6.000000e+10,1200000000,5.644579e+09
1,شایان,شایان یک,جعاله,واریز به حساب,3000000000,1.0,23,12,50.0,A,None,1.500000e+10,100000000,1,غیر حضوری/ حضوری,None,6.000000e+10,1200000000,5.644579e+09
2,شایان,شایان یک,مرابحه,واریز به حساب,3000000000,1.0,23,12,50.0,B,None,1.500000e+10,100000000,1,غیر حضوری/ حضوری,None,6.000000e+10,1200000000,5.644579e+09
3,شایان,شایان یک,جعاله,واریز به حساب,3000000000,1.0,23,12,50.0,B,None,1.500000e+10,100000000,1,غیر حضوری/ حضوری,None,6.000000e+10,1200000000,5.644579e+09
4,شایان,شایان یک,مرابحه,واریز به حساب,3000000000,2.0,23,12,150.0,A,None,1.500000e+10,100000000,1,غیر حضوری/ حضوری,None,1.800000e+11,1200000000,1.693374e+10
